# Reuters 뉴스 기사 분류 실습 (개선 버전)

## 주요 개선 사항

| 항목 | 기존 | 개선 | 이유 |
|------|------|------|------|
| vocab_size | 1,000 | 10,000 | 단어의 97% 가 `<unk>` 처리되던 문제 해결 |
| max_len | 100 | 200 | 평균 기사 길이 145토큰을 충분히 커버 |
| LSTM 방향 | 단방향 | 양방향(Bidirectional) | 문맥 이해력 향상 |
| LSTM 레이어 | 1층 | 2층 | 표현력 향상 |
| 출력 추출 | 마지막 hidden state | Global Max Pooling | 전체 시퀀스에서 중요 특징 추출 |
| 클래스 불균형 | 미처리 | class_weight 적용 | 소수 클래스 학습 개선 |
| 검증 방법 | 테스트셋으로 검증 | 훈련셋에서 10% 분리 | 데이터 누수 방지 |
| 학습률 | 고정 0.001 | ReduceLROnPlateau 스케줄러 | 정체 구간 자동 조정 |
| Gradient Clipping | 없음 | max_norm=1.0 | LSTM 폭발 기울기 방지 |
| 학습 에포크 | 5 | 30 (Early Stopping) | 충분한 학습 기회 제공 |

## 1. 기본 라이브러리 불러오기

In [ ]:
import os
import time
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split

print("PyTorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

## 2. 난수 고정 함수

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 3. Reuters 뉴스 기사 데이터셋 불러오기

In [ ]:
def load_reuters_dataset(num_words=None, test_split=0.2):
    try:
        from tensorflow.keras.datasets import reuters
        (X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = reuters.load_data(
            num_words=num_words,
            test_split=test_split
        )
        word_to_index = reuters.get_word_index()
        print("TensorFlow/Keras Reuters 데이터셋을 불러왔습니다.")
        return (X_train_raw, y_train_raw), (X_test_raw, y_test_raw), word_to_index

    except Exception as error:
        print("Reuters 데이터셋 로딩에 실패하여 예제용 작은 데이터를 생성합니다.")
        print("오류 내용:", error)
        word_to_index = {
            "market": 1, "stock": 2, "profit": 3, "company": 4, "oil": 5,
            "trade": 6, "bank": 7, "money": 8, "team": 9, "game": 10,
            "win": 11, "minister": 12, "government": 13, "policy": 14,
            "technology": 15, "computer": 16, "software": 17
        }
        X_train_raw = [
            [1, 2, 3, 4, 8], [5, 6, 1, 2], [9, 10, 11], [12, 13, 14],
            [15, 16, 17], [1, 7, 8, 3], [9, 11, 10], [13, 12, 14]
        ]
        y_train_raw = np.array([0, 0, 1, 2, 3, 0, 1, 2], dtype=np.int64)
        X_test_raw = [[2, 3, 4], [10, 11, 9], [16, 17, 15], [12, 14, 13]]
        y_test_raw = np.array([0, 1, 3, 2], dtype=np.int64)
        return (X_train_raw, y_train_raw), (X_test_raw, y_test_raw), word_to_index


(X_train_raw, y_train_raw), (X_test_raw, y_test_raw), word_to_index = load_reuters_dataset(
    num_words=None,
    test_split=0.2
)

print("훈련용 뉴스 기사 수:", len(X_train_raw))
print("테스트용 뉴스 기사 수:", len(X_test_raw))
num_classes_raw = len(set(np.concatenate([np.array(y_train_raw), np.array(y_test_raw)])))
print("전체 뉴스 카테고리 수:", num_classes_raw)

## 4. 뉴스 기사 데이터 구조 확인

In [ ]:
print("첫 번째 훈련용 뉴스 기사 정수 시퀀스:")
print(X_train_raw[0])
print("\n첫 번째 훈련용 뉴스 기사의 라벨:", y_train_raw[0])
print("첫 번째 훈련용 뉴스 기사의 길이:", len(X_train_raw[0]))

train_lengths = [len(s) for s in X_train_raw]
print("\n기사 길이 통계:")
print("  최대 길이:", max(train_lengths))
print("  평균 길이: {:.2f}".format(np.mean(train_lengths)))
print("  중앙값 길이:", int(np.median(train_lengths)))
print("  75th 퍼센타일:", int(np.percentile(train_lengths, 75)))

## 5. 라벨 분포 확인 (클래스 불균형 파악)

In [ ]:
unique_labels, label_counts = np.unique(y_train_raw, return_counts=True)

print("상위 10개 빈도 라벨:")
sorted_idx = np.argsort(label_counts)[::-1]
for i in sorted_idx[:10]:
    print(f"  라벨 {unique_labels[i]:2d}: {label_counts[i]:4d}건")

print("\n하위 10개 빈도 라벨:")
for i in sorted_idx[-10:]:
    print(f"  라벨 {unique_labels[i]:2d}: {label_counts[i]:4d}건")

print(f"\n최대/최소 비율: {label_counts.max()/label_counts.min():.1f}배 (클래스 불균형 심각)")

## 6. 하이퍼파라미터 설정 (개선)

**핵심 변경:**
- `vocab_size`: 1,000 → **10,000** (가장 중요한 변경)
- `max_len`: 100 → **200** (평균 길이 145를 커버)
- `epochs`: 5 → **30** (Early Stopping으로 자동 종료)

In [ ]:
# [개선] vocab_size를 10,000으로 증가 (기존 1,000은 어휘의 97%가 <unk> 처리됨)
vocab_size = 10000

# [개선] max_len을 200으로 증가 (기존 100은 평균 기사 길이 145.5를 초과하여 잘림)
max_len = 200

# [개선] embedding_dim 증가로 더 풍부한 단어 표현
embedding_dim = 128

# [개선] hidden_units 증가
hidden_units = 128

# [신규] LSTM 레이어 수 (다층 구조로 표현력 향상)
num_layers = 2

# [신규] 양방향 LSTM 사용 여부
bidirectional = True

# [개선] 다층/양방향 모델에 맞게 dropout 증가
dropout_rate = 0.5

# 배치 크기
batch_size = 64

# [개선] 에포크 수 증가 (Early Stopping이 자동으로 종료)
epochs = 30

# 학습률
learning_rate = 0.001

# [개선] patience 증가 (LR 스케줄러와 함께 사용)
patience = 5

print(f"vocab_size   : {vocab_size:,} (기존: 1,000)")
print(f"max_len      : {max_len} (기존: 100)")
print(f"embedding_dim: {embedding_dim}")
print(f"hidden_units : {hidden_units}")
print(f"num_layers   : {num_layers} (기존: 1)")
print(f"bidirectional: {bidirectional} (기존: False)")
print(f"dropout_rate : {dropout_rate}")
print(f"batch_size   : {batch_size}")
print(f"epochs       : {epochs} (기존: 5)")

## 7. 패딩 함수

In [ ]:
def pad_sequences_torch_style(sequences, maxlen, padding="pre", truncating="pre", value=0):
    padded_sequences = []
    for sequence in sequences:
        sequence = list(sequence)
        if len(sequence) > maxlen:
            sequence = sequence[-maxlen:] if truncating == "pre" else sequence[:maxlen]
        pad_length = maxlen - len(sequence)
        if padding == "pre":
            padded_sequence = [value] * pad_length + sequence
        else:
            padded_sequence = sequence + [value] * pad_length
        padded_sequences.append(padded_sequence)
    return np.array(padded_sequences, dtype=np.int64)

## 8. 모델 입력 데이터 준비

In [ ]:
(X_train, y_train), (X_test, y_test), word_to_index = load_reuters_dataset(
    num_words=vocab_size,
    test_split=0.2
)

X_train = pad_sequences_torch_style(X_train, maxlen=max_len, padding="pre", truncating="pre", value=0)
X_test  = pad_sequences_torch_style(X_test,  maxlen=max_len, padding="pre", truncating="pre", value=0)

y_train = np.array(y_train, dtype=np.int64)
y_test  = np.array(y_test,  dtype=np.int64)

num_classes = int(max(np.max(y_train), np.max(y_test)) + 1)

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
print("num_classes:  ", num_classes)

## 9. 클래스 불균형 해소 - 클래스 가중치 계산

라벨 3은 3,159개, 라벨 35는 10개로 316배 불균형.
CrossEntropyLoss에 class_weight를 적용하여 소수 클래스에 더 높은 가중치를 부여한다.

In [ ]:
# 각 클래스의 샘플 수를 카운트한다.
class_counts = np.bincount(y_train, minlength=num_classes).astype(np.float32)

# 샘플 수가 0인 클래스는 1로 대체하여 나눗셈 오류를 방지한다.
class_counts = np.where(class_counts == 0, 1.0, class_counts)

# 역수를 취해 소수 클래스에 높은 가중치를 부여한다.
class_weights = 1.0 / class_counts

# 가중치 합이 num_classes가 되도록 정규화한다.
class_weights = class_weights / class_weights.sum() * num_classes

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print("클래스 가중치 (상위 5개):")
top5 = np.argsort(class_weights)[::-1][:5]
for i in top5:
    print(f"  클래스 {i:2d}: 가중치 {class_weights[i]:.4f} (샘플 수 {int(class_counts[i])}개)")

## 10. Tensor 변환 및 Dataset/DataLoader 생성

[개선] 훈련셋의 10%를 검증셋으로 분리 (기존: 테스트셋으로 검증하여 데이터 누수 발생)

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor  = torch.tensor(X_test,  dtype=torch.long)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.long)

full_train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset       = TensorDataset(X_test_tensor,  y_test_tensor)

# 훈련셋의 10%를 검증셋으로 분리한다.
val_size   = int(0.1 * len(full_train_dataset))
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

print(f"훈련 샘플 수   : {len(train_dataset):,}")
print(f"검증 샘플 수   : {len(val_dataset):,}")
print(f"테스트 샘플 수 : {len(test_dataset):,}")

## 11. 개선된 양방향 LSTM 모델 구현

**변경 사항:**
- 단방향 → **양방향 LSTM** (앞뒤 문맥 모두 학습)
- 1층 → **2층 LSTM** (더 깊은 표현 학습)
- 마지막 hidden state → **Global Max Pooling** (전체 시퀀스에서 가장 중요한 특징 추출)

In [ ]:
class ReutersLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_units, num_classes,
                 dropout_rate, num_layers=2, bidirectional=True):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        # [개선] 양방향 2층 LSTM
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_units,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_rate if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )

        self.dropout = nn.Dropout(p=dropout_rate)

        # 양방향이면 hidden_units * 2, 단방향이면 hidden_units
        lstm_output_dim = hidden_units * 2 if bidirectional else hidden_units

        self.fc = nn.Linear(in_features=lstm_output_dim, out_features=num_classes)

    def forward(self, x):
        # [배치, 문장길이] → [배치, 문장길이, 임베딩]
        embedded = self.embedding(x)

        # lstm_output: [배치, 문장길이, hidden*방향수]
        lstm_output, _ = self.lstm(embedded)

        # [개선] Global Max Pooling: 전체 시퀀스에서 각 특징의 최댓값을 추출
        # 기존의 마지막 hidden state보다 더 많은 정보를 보존
        pooled = torch.max(lstm_output, dim=1)[0]

        dropped = self.dropout(pooled)
        logits = self.fc(dropped)
        return logits


model = ReutersLSTMClassifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_units=hidden_units,
    num_classes=num_classes,
    dropout_rate=dropout_rate,
    num_layers=num_layers,
    bidirectional=bidirectional
).to(device)

print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n학습 가능한 파라미터 수: {total_params:,}")

## 12. 손실 함수, 최적화 알고리즘, 학습률 스케줄러 설정

In [ ]:
# [개선] 클래스 가중치를 적용하여 불균형 데이터 문제를 완화한다.
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# [신규] 검증 손실이 2 에포크 동안 개선 없으면 학습률을 절반으로 줄인다.
# verbose 파라미터는 PyTorch 2.2+에서 deprecated되어 제거한다.
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=2,
    factor=0.5
)

os.makedirs("models", exist_ok=True)
best_model_path = os.path.join("models", "best_reuters_lstm_improved.pt")
best_val_loss = float("inf")
epochs_without_improvement = 0

print("손실 함수:", criterion)
print("최적화 알고리즘:", optimizer.__class__.__name__)
print("학습률 스케줄러:", scheduler.__class__.__name__)

## 13. 정확도 계산 함수

In [ ]:
def calculate_accuracy(logits, labels):
    predictions = torch.argmax(logits, dim=1)
    correct = (predictions == labels).sum().item()
    total = labels.size(0)
    return correct / total

## 14. 학습 함수 (Gradient Clipping 추가)

In [ ]:
def train_one_epoch(model, data_loader, criterion, optimizer, device, max_grad_norm=1.0):
    model.train()
    total_loss = 0.0
    total_accuracy = 0.0

    for batch_inputs, batch_labels in data_loader:
        batch_inputs = batch_inputs.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()
        logits = model(batch_inputs)
        loss = criterion(logits, batch_labels)
        loss.backward()

        # [신규] Gradient Clipping: LSTM의 폭발 기울기 문제를 방지한다.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)

        optimizer.step()

        total_loss     += loss.item() * batch_inputs.size(0)
        total_accuracy += calculate_accuracy(logits, batch_labels) * batch_inputs.size(0)

    avg_loss     = total_loss     / len(data_loader.dataset)
    avg_accuracy = total_accuracy / len(data_loader.dataset)
    return avg_loss, avg_accuracy

## 15. 평가 함수

In [ ]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_accuracy = 0.0

    with torch.no_grad():
        for batch_inputs, batch_labels in data_loader:
            batch_inputs = batch_inputs.to(device)
            batch_labels = batch_labels.to(device)
            logits = model(batch_inputs)
            loss   = criterion(logits, batch_labels)

            total_loss     += loss.item() * batch_inputs.size(0)
            total_accuracy += calculate_accuracy(logits, batch_labels) * batch_inputs.size(0)

    avg_loss     = total_loss     / len(data_loader.dataset)
    avg_accuracy = total_accuracy / len(data_loader.dataset)
    return avg_loss, avg_accuracy

## 16. 모델 학습 실행

In [ ]:
train_losses      = []
val_losses        = []
train_accuracies  = []
val_accuracies    = []
lr_history        = []

start_time = time.time()

for epoch in range(1, epochs + 1):
    train_loss, train_accuracy = train_one_epoch(
        model=model,
        data_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        max_grad_norm=1.0
    )

    # [개선] 테스트셋이 아닌 별도 검증셋으로 평가
    val_loss, val_accuracy = evaluate(
        model=model,
        data_loader=val_loader,
        criterion=criterion,
        device=device
    )

    # [신규] LR 스케줄러에 검증 손실을 전달한다.
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)
    lr_history.append(current_lr)

    print(
        f"Epoch [{epoch:2d}/{epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_accuracy:.4f} "
        f"LR: {current_lr:.6f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "vocab_size": vocab_size,
                "embedding_dim": embedding_dim,
                "hidden_units": hidden_units,
                "num_classes": num_classes,
                "dropout_rate": dropout_rate,
                "max_len": max_len,
                "num_layers": num_layers,
                "bidirectional": bidirectional
            },
            best_model_path
        )
        print(f"  → 모델 저장 (Val Loss: {best_val_loss:.4f})")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f"\nEarly Stopping: {patience} 에포크 동안 개선 없음. 학습 중단.")
            break

end_time = time.time()
print(f"\n전체 학습 시간: {end_time - start_time:.2f}초")

## 17. 학습 결과 시각화

In [ ]:
epochs_range = range(1, len(train_accuracies) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs_range, train_accuracies, label="Train Accuracy")
axes[0].plot(epochs_range, val_accuracies,   label="Validation Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy")
axes[0].legend()

axes[1].plot(epochs_range, train_losses, label="Train Loss")
axes[1].plot(epochs_range, val_losses,   label="Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Loss")
axes[1].legend()

axes[2].plot(epochs_range, lr_history, label="Learning Rate", color="green")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Learning Rate")
axes[2].set_title("Learning Rate Schedule")
axes[2].legend()

plt.tight_layout()
plt.show()

## 18. 저장된 최적 모델 불러오기

In [ ]:
if os.path.exists(best_model_path):
    # [개선] weights_only=False 명시로 경고 제거 (PyTorch 2.x 이상)
    checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)

    loaded_model = ReutersLSTMClassifier(
        vocab_size=checkpoint["vocab_size"],
        embedding_dim=checkpoint["embedding_dim"],
        hidden_units=checkpoint["hidden_units"],
        num_classes=checkpoint["num_classes"],
        dropout_rate=checkpoint["dropout_rate"],
        num_layers=checkpoint["num_layers"],
        bidirectional=checkpoint["bidirectional"]
    ).to(device)

    loaded_model.load_state_dict(checkpoint["model_state_dict"])
    print("저장된 최적 모델을 불러왔습니다.")
else:
    loaded_model = model
    print("저장된 모델이 없어 현재 모델을 사용합니다.")

## 19. 테스트 데이터 최종 평가

In [ ]:
# 검증셋 평가
val_loss_final, val_accuracy_final = evaluate(
    model=loaded_model,
    data_loader=val_loader,
    criterion=criterion,
    device=device
)

# 테스트셋 평가 (최종 성능)
test_loss, test_accuracy = evaluate(
    model=loaded_model,
    data_loader=test_loader,
    criterion=criterion,
    device=device
)

print(f"검증  손실: {val_loss_final:.4f}  |  검증  정확도: {val_accuracy_final:.4f} ({val_accuracy_final*100:.2f}%)")
print(f"테스트 손실: {test_loss:.4f}  |  테스트 정확도: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print("\n※ 기존 모델 테스트 정확도: 55.57% → 개선 목표: 70% 이상")

## 20. 테스트 기사 예측

In [ ]:
sample_article = X_test_tensor[0:1]
true_label = int(y_test_tensor[0].item())

loaded_model.eval()
with torch.no_grad():
    logits = loaded_model(sample_article.to(device))
    probabilities = torch.softmax(logits, dim=1)
    predicted_label = int(torch.argmax(probabilities, dim=1).item())
    predicted_probability = float(torch.max(probabilities).item())

print("실제 라벨:", true_label)
print("예측 라벨:", predicted_label)
print("예측 확률: {:.4f}".format(predicted_probability))
print("예측 결과:", "정답" if predicted_label == true_label else "오답")

## 21. 새 영어 문장 입력 후 카테고리 예측

In [ ]:
def encode_new_text(text, word_to_index, vocab_size=10000, max_len=200):
    words = text.lower().split()
    encoded = [1]  # <sos> 토큰
    for word in words:
        index = word_to_index.get(word, 2) + 3
        encoded.append(index if index < vocab_size else 2)
    padded = pad_sequences_torch_style(
        [encoded], maxlen=max_len, padding="pre", truncating="pre", value=0
    )
    return torch.tensor(padded, dtype=torch.long)


def predict_news_category(text, model, word_to_index, vocab_size=10000, max_len=200, device="cpu"):
    input_tensor = encode_new_text(text, word_to_index, vocab_size, max_len).to(device)
    model.eval()
    with torch.no_grad():
        probabilities = torch.softmax(model(input_tensor), dim=1)
        predicted_label = int(torch.argmax(probabilities, dim=1).item())
        predicted_probability = float(torch.max(probabilities).item())
    return predicted_label, predicted_probability


# 예측 테스트 (경제 뉴스)
test_texts = [
    "stock market profit company trade bank money",
    "oil production energy price barrel",
    "government minister policy tax budget"
]

for text in test_texts:
    label, prob = predict_news_category(
        text=text,
        model=loaded_model,
        word_to_index=word_to_index,
        vocab_size=vocab_size,
        max_len=max_len,
        device=device
    )
    print(f"입력: '{text}'")
    print(f"  → 예측 라벨: {label}, 확률: {prob:.4f}\n")

## 마무리 정리

### 개선 전후 비교

| 구분 | 기존 | 개선 |
|------|------|------|
| 테스트 정확도 | 55.57% | 70%+ 목표 |
| vocab_size | 1,000 | 10,000 |
| max_len | 100 | 200 |
| LSTM | 단방향 1층 | 양방향 2층 |
| 출력 방식 | 마지막 hidden | Global Max Pooling |
| 클래스 불균형 | 미처리 | class_weight 적용 |
| 검증 방식 | 테스트셋 사용 | 별도 검증셋 분리 |
| LR 스케줄러 | 없음 | ReduceLROnPlateau |
| Gradient Clipping | 없음 | max_norm=1.0 |